# Turbofan RUL Tahmini — Bölüm 4
## Hiperparametre Optimizasyonu ve Güçlü Rakiplerle Karşılaştırma (Aşama 10–11)

Bu notebook iki aşamayı içerir:
- **Aşama 10:** Bölüm 3'te seçilen aday model **HistGradientBoosting** için hiperparametre araması. Grid Search, Random Search ve Bayesian (Optuna TPE) yöntemleri karşılaştırılır; tuning öncesi ve sonrası sonuçlar gösterilir.
- **Aşama 11:** Optimize edilmiş aday, **Random Forest, SVR, LightGBM ve XGBoost** ile **aynı tuning bütçesi ve stratejisiyle** karşılaştırılır ve final model seçilir.

> ⚠️ Yalnızca `train_FD001.txt` kullanılır. Test dosyaları Aşama 12'ye kadar açılmaz.

> ⏱️ **Toplam süre (ücretsiz Colab):** ≈ 50–70 dk (`FAST_MODE = True` ile ≈ 25–35 dk). Notebook **kaldığı yerden devam eder**: her deneme Drive'a kaydedilir. Oturum koparsa Drive bağlantı hücresinden itibaren tekrar **Tümünü çalıştır** demeniz yeterli; tamamlanan denemeler tekrar yapılmaz.
>
> Ücretsiz Colab, sekmeyle uzun süre etkileşim olmazsa oturumu kapatabilir. Uzun aramalar sırasında sekmeyi açık tutun.

## 0. Kurulum
### 0.1 Drive bağlantısı

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/BTK_RUL"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BASE_DIR, exist_ok=True)
    os.chdir(BASE_DIR)
except ImportError:
    print("Colab dışında çalışılıyor → mevcut klasör kullanılacak.")
print("Çalışma klasörü:", os.getcwd())
if not os.path.exists("train_FD001.txt"):
    raise FileNotFoundError("train_FD001.txt bulunamadı. Dosyayı Drive'daki BTK_RUL klasörüne yükleyin.")
os.makedirs("checkpoints", exist_ok=True)

### 0.2 Kütüphaneler
Optuna Colab'da hazır gelmez; LightGBM ve XGBoost genellikle yüklüdür ama sürüm uyumu için hepsi birlikte kurulur.

In [ ]:
%pip install -q optuna lightgbm xgboost

In [ ]:
%%writefile rul_utils.py
"""rul_utils.py — NASA C-MAPSS RUL projesi için ortak yardımcılar.
Notebook'lar ve Streamlit uygulaması aynı kodu kullanır (tekrarlanabilirlik)."""
import time
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

INDEX_COLS = ["unit", "cycle"]
SETTING_COLS = ["op1", "op2", "op3"]
SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
COLS = INDEX_COLS + SETTING_COLS + SENSOR_COLS
FEATURES_14 = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12",
               "s13", "s14", "s15", "s17", "s20", "s21"]


def load_cmapss(path):
    """Boşlukla ayrılmış, başlıksız C-MAPSS dosyasını okur."""
    df = pd.read_csv(path, sep=r"\s+", header=None).iloc[:, :26]
    df.columns = COLS
    return df.sort_values(["unit", "cycle"]).reset_index(drop=True)


def add_rul(df):
    """Eğitim verisi için gerçek (kırpılmamış) RUL: motorun son döngüsü - mevcut döngü."""
    df = df.copy()
    df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
    return df


def _rolling_slope(x, window):
    """Geçmiş `window` döngüdeki doğrusal eğim (yalnızca geçmişe bakar)."""
    s = pd.Series(np.asarray(x, dtype=float))
    t = pd.Series(np.arange(len(s), dtype=float))
    cov = s.rolling(window, min_periods=2).cov(t)
    var = t.rolling(window, min_periods=2).var()
    return (cov / var).fillna(0.0).to_numpy()


class RollingFeatures(BaseEstimator, TransformerMixin):
    """Motor bazında, yalnızca GEÇMİŞ döngüleri kullanan kayan pencere özellikleri.

    - Girdi: 'unit', 'cycle' ve sensör sütunlarını içeren DataFrame (motor içinde döngü sırasına göre).
    - Çıktı: ham sensörler (+ isteğe bağlı) kayan ortalama, standart sapma ve eğim.
    - 'unit' ve 'cycle' çıktıya ALINMAZ (özellik değil, yalnızca gruplama/sıra bilgisi).
    - fit() hiçbir şey öğrenmez (stateless): bir satırın özellikleri yalnızca kendi motorunun
      o ana kadarki kayıtlarından hesaplanır → başka motorlardan veya gelecekten bilgi sızmaz.
    """

    def __init__(self, sensors=tuple(FEATURES_14), window=10, stats=("mean", "std", "slope"), keep_raw=True):
        self.sensors = sensors
        self.window = window
        self.stats = stats
        self.keep_raw = keep_raw

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        if not X.groupby("unit")["cycle"].apply(lambda c: c.is_monotonic_increasing).all():
            raise ValueError("Her motorun satırları döngü sırasına göre sıralı olmalı.")
        sensors, w = list(self.sensors), int(self.window)
        g = X.groupby("unit", sort=False)
        parts = []
        if self.keep_raw:
            parts.append(X[sensors].reset_index(drop=True))
        if "mean" in self.stats:
            m = g[sensors].transform(lambda c: c.rolling(w, min_periods=1).mean())
            parts.append(m.add_suffix(f"_mean{w}").reset_index(drop=True))
        if "std" in self.stats:
            sd = g[sensors].transform(lambda c: c.rolling(w, min_periods=2).std()).fillna(0.0)
            parts.append(sd.add_suffix(f"_std{w}").reset_index(drop=True))
        if "slope" in self.stats:
            sl = g[sensors].transform(lambda c: _rolling_slope(c, w))
            parts.append(sl.add_suffix(f"_slope{w}").reset_index(drop=True))
        out = pd.concat(parts, axis=1)
        out.index = X.index
        return out

    def get_feature_names_out(self, input_features=None):
        names = list(self.sensors) if self.keep_raw else []
        for st in ("mean", "std", "slope"):
            if st in self.stats:
                names += [f"{s}_{st}{self.window}" for s in self.sensors]
        return np.array(names)


def nasa_score(y_true, y_pred):
    """PHM08 skor fonksiyonu. d = tahmin - gerçek.
    Geç tahmin (d > 0, ömrü fazla tahmin etmek) daha ağır cezalandırılır."""
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1)))


def regression_metrics(y_true, y_pred, prefix=""):
    return {
        f"{prefix}RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}R2": float(r2_score(y_true, y_pred)),
        f"{prefix}NASA": nasa_score(y_true, y_pred),
    }


def test_like_rows(df, idx, n_cuts=10, min_cycle=31, max_rul=150, seed=42):
    """Validation motorlarından resmi test protokolünü taklit eden satırlar seçer:
    her motor için rastgele `n_cuts` kesim noktası; kesim anındaki GERÇEK RUL ile değerlendirilir.
    - min_cycle: kayan pencerelerin dolması için gereken minimum geçmiş.
    - max_rul: bakım karar ufku. Kesim noktaları gerçek RUL <= max_rul olan döngülerden seçilir
      (EDA: bozulma ömrün ikinci yarısında görünür hale geliyor; daha erken döngülerde RUL
      sensörlerden ayırt edilemez ve bakım kararı açısından önemsizdir)."""
    rng = np.random.default_rng(seed)
    sub = df.loc[idx]
    chosen = []
    for _, d in sub.groupby("unit"):
        cand = d.index[(d["cycle"] >= min(min_cycle, d["cycle"].max())) & (d["RUL"] <= max_rul)]
        chosen.extend(rng.choice(cand, size=min(n_cuts, len(cand)), replace=False))
    return np.sort(np.array(chosen))


def cv_evaluate(model, df, cap, cv, feature_cols=None, n_cuts=10, max_rul=150, seed=42, name=None,
                train_metrics=False, model_size=False, return_preds=False):
    """Motor bazlı CV ile değerlendirme.
    - Eğitim hedefi: RUL üst sınırla kırpılır (cap=None ise kırpılmaz).
    - 'clip_' metrikleri: validation motorlarının tüm döngüleri, kırpılmış hedefe göre.
    - 'test_' metrikleri: validation motorlarında rastgele kesim noktaları, GERÇEK RUL'a göre
      ve bakım karar ufku (gerçek RUL <= max_rul) içinde → model ve cap seçiminde birincil ölçüt.
    - train_metrics=True: eğitim fold'undaki RMSE (aşırı öğrenme kontrolü).
    - model_size=True: eğitilmiş modelin bellekteki (pickle) boyutu, MB.
    - return_preds=True: (sonuç_tablosu, kesim_noktası_tahminleri) döndürür (hata analizi için).
    """
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_true = df["RUL"].to_numpy()
    y_fit = y_true if cap is None else np.minimum(y_true, cap)
    rows, preds = [], []
    for k, (tr, va) in enumerate(cv.split(X, y_fit, groups=df["unit"])):
        m = clone(model)
        t0 = time.perf_counter()
        m.fit(X.iloc[tr], y_fit[tr])
        fit_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        p = np.clip(m.predict(X.iloc[va]), 0, None)
        pred_s = time.perf_counter() - t0
        res = {"fold": k + 1, "fit_s": fit_s, "pred_s": pred_s}
        res.update(regression_metrics(y_fit[va], p, "clip_"))
        cut = test_like_rows(df, df.index[va], n_cuts=n_cuts, max_rul=max_rul, seed=seed + k)
        pos = pd.Series(np.arange(len(va)), index=df.index[va]).loc[cut].to_numpy()
        res.update(regression_metrics(y_true[cut], p[pos], "test_"))
        # Arızaya yakın bölge (gerçek RUL <= 50): bakım kararının en kritik olduğu dönem
        near = y_true[cut] <= 50
        res["near50_RMSE"] = float(np.sqrt(np.mean((p[pos][near] - y_true[cut][near]) ** 2))) if near.any() else np.nan
        # Geç tahmin oranı: ömrü 10 döngüden fazla FAZLA tahmin etme (güvenlik açısından tehlikeli)
        res["late10_pct"] = float(np.mean(p[pos] - y_true[cut] > 10) * 100)
        if train_metrics:
            p_tr = np.clip(m.predict(X.iloc[tr]), 0, None)
            res["train_clip_RMSE"] = float(np.sqrt(mean_squared_error(y_fit[tr], p_tr)))
        if model_size:
            import pickle
            res["size_MB"] = len(pickle.dumps(m)) / 1e6
        rows.append(res)
        if return_preds:
            preds.append(pd.DataFrame({"model": name, "fold": k + 1, "unit": df.loc[cut, "unit"].to_numpy(),
                                       "cycle": df.loc[cut, "cycle"].to_numpy(),
                                       "RUL_true": y_true[cut], "RUL_pred": p[pos]}))
    out = pd.DataFrame(rows)
    if name is not None:
        out.insert(0, "model", name)
    if return_preds:
        return out, pd.concat(preds, ignore_index=True)
    return out


def summarize(cv_df, by="model"):
    cols = [c for c in ["test_RMSE", "test_MAE", "test_R2", "test_NASA", "near50_RMSE", "late10_pct",
                        "clip_RMSE", "train_clip_RMSE", "fit_s", "pred_s", "size_MB"] if c in cv_df.columns]
    agg = cv_df.groupby(by)[cols].agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    return agg.sort_values("test_RMSE_mean")


def oof_predictions(model, df, cap, cv, feature_cols=None):
    """Out-of-fold tahminler: her satır, motoru validation'dayken eğitilen modelden tahmin edilir."""
    feature_cols = feature_cols or (INDEX_COLS + FEATURES_14)
    X = df[feature_cols]
    y_fit = df["RUL"].to_numpy() if cap is None else np.minimum(df["RUL"].to_numpy(), cap)
    oof = np.zeros(len(df))
    for tr, va in cv.split(X, y_fit, groups=df["unit"]):
        m = clone(model).fit(X.iloc[tr], y_fit[tr])
        oof[va] = np.clip(m.predict(X.iloc[va]), 0, None)
    return oof

In [ ]:
import warnings, json, time, platform, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn, optuna, lightgbm, xgboost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from rul_utils import *

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
            "optuna": optuna.__version__, "lightgbm": lightgbm.__version__, "xgboost": xgboost.__version__}
print(VERSIONS)

### 0.3 Ayarlar ve önceki bölümlerin kararları

In [ ]:
DEFAULT_CONFIG = {"seed": 42, "cap": 125, "window": 30, "stats": ["mean", "std", "slope"], "eval_max_rul": 150, "eval_n_cuts": 10}
config = {**DEFAULT_CONFIG, **(json.load(open("project_config.json")) if os.path.exists("project_config.json") else {})}
SEED, CAP = config["seed"], config["cap"]
STATS = tuple(config["stats"])
np.random.seed(SEED)

# ---- Tuning bütçesi (tüm modeller için AYNI) ----
FAST_MODE = False                          # True: daha kısa deneme (≈ yarı süre), sonuçlar biraz daha az güvenilir
N_TRIALS = 15 if FAST_MODE else 30         # model başına en fazla deneme
TIMEOUT_MIN = 8 if FAST_MODE else 15       # model başına en fazla süre (dakika)
WINDOWS = [30, 40, 50, 60, 70, 90]         # kayan pencere uzunluğu da arama uzayında

df = add_rul(load_cmapss("train_FD001.txt"))
cv = GroupKFold(n_splits=5)
print(f"cap={CAP} | bütçe: {N_TRIALS} deneme veya {TIMEOUT_MIN} dk / model | pencereler={WINDOWS}")
print("Aday (Bölüm 3):", config.get("candidate_model", "HistGradientBoostingRegressor"))

### 0.4 Özellik önbelleği (hız optimizasyonu)

Her denemede kayan pencere özelliklerini yeniden hesaplamak yerine, her pencere uzunluğu için **bir kez** hesaplayıp saklıyoruz.

**Bu sızıntı yaratır mı? Hayır.** Bölüm 2'de (5.4) iki testle gösterdik: `RollingFeatures` durumsuz (stateless). Bir satırın özellikleri yalnızca **kendi motorunun geçmiş döngülerinden** hesaplanıyor; başka motorlardan ya da gelecekten bilgi almıyor. Bu yüzden özellikleri CV'den önce bir kez hesaplamak, her fold içinde ayrı ayrı hesaplamakla **birebir aynı** sonucu verir. Veriden istatistik öğrenen adımlar (`StandardScaler` ve hedef ölçekleyici) ise hâlâ Pipeline içinde ve **her fold'un eğitim verisinde ayrı ayrı** fit ediliyor.

In [ ]:
FEAT, FEAT_COLS = {}, {}
for w in WINDOWS:
    F = RollingFeatures(window=w, stats=STATS).transform(df[INDEX_COLS + FEATURES_14])
    FEAT[w] = pd.concat([df[["unit", "cycle", "RUL"]], F], axis=1)
    FEAT_COLS[w] = list(F.columns)
print({w: len(c) for w, c in FEAT_COLS.items()})

# Doğrulama: önbellekli değerlendirme == Pipeline içinde hesaplama (Bölüm 3 sonucu)
def make_cached(estimator):
    """Önbellekli özellikler için pipeline: ölçekleme → model (hedef de ölçeklenir, Bölüm 3 ile aynı)."""
    return TransformedTargetRegressor(regressor=Pipeline([("scaler", StandardScaler()), ("model", estimator)]),
                                      transformer=StandardScaler())

def make_full(estimator, window):
    """Ham sensör kaydından başlayan tam pipeline (final model ve Streamlit için)."""
    return TransformedTargetRegressor(regressor=Pipeline([("features", RollingFeatures(window=window, stats=STATS)),
                                                          ("scaler", StandardScaler()), ("model", estimator)]),
                                      transformer=StandardScaler())

def evaluate_config(estimator, window, seed=SEED, n_cuts=None, **kw):
    return cv_evaluate(make_cached(estimator), FEAT[window], CAP, cv, feature_cols=FEAT_COLS[window],
                       n_cuts=n_cuts or config["eval_n_cuts"], max_rul=config["eval_max_rul"], seed=seed, **kw)

a = evaluate_config(HistGradientBoostingRegressor(random_state=SEED), 30)["test_RMSE"].mean()
b = cv_evaluate(make_full(HistGradientBoostingRegressor(random_state=SEED), 30), df, CAP, cv, seed=SEED)["test_RMSE"].mean()
print(f"Önbellekli: {a:.4f} | Pipeline içinde: {b:.4f} | aynı mı: {np.isclose(a, b)}")

### 0.5 Ortak tuning altyapısı

Tüm modeller **aynı fonksiyonla** optimize edilir. Böylece Aşama 11'deki "benzer tuning bütçesi ve stratejisi" şartı kodla garanti altına alınır:
- **Amaç fonksiyonu:** 5-fold GroupKFold test-benzeri **RMSE ortalaması** (minimize). NASA skoru, RUL ≤ 50 hatası, geç tahmin oranı ve süre her deneme için ayrıca kaydedilir.
- **Bütçe:** model başına `N_TRIALS` deneme veya `TIMEOUT_MIN` dakika (hangisi önce dolarsa).
- **Kaldığı yerden devam:** Her deneme sonrası çalışma `checkpoints/` klasörüne kaydedilir.
- **Hata toleransı:** Geçersiz bir parametre kombinasyonu hata verirse o deneme atlanır, arama durmaz.

In [ ]:
EXTRA_ATTRS = ["test_NASA", "near50_RMSE", "late10_pct", "clip_RMSE", "fit_s"]

def make_objective(space_fn):
    def objective(trial):
        estimator, window = space_fn(trial)
        r = evaluate_config(estimator, window)
        for k in EXTRA_ATTRS:
            trial.set_user_attr(k, float(r[k].mean()))
        trial.set_user_attr("test_RMSE_std", float(r["test_RMSE"].std()))
        return float(r["test_RMSE"].mean())
    return objective

def completed(study):
    return [t for t in study.trials if t.value is not None]

def run_study(name, space_fn, sampler="tpe", n_trials=None, timeout_min=None):
    """Optuna çalışmasını başlatır ya da checkpoint'ten devam ettirir."""
    n_trials, timeout_min = n_trials or N_TRIALS, timeout_min or TIMEOUT_MIN
    path = f"checkpoints/study_{name}.pkl"
    if os.path.exists(path):
        study = joblib.load(path)
    else:
        smp = optuna.samplers.TPESampler(seed=SEED) if sampler == "tpe" else optuna.samplers.RandomSampler(seed=SEED)
        study = optuna.create_study(direction="minimize", sampler=smp, study_name=name)
        study.set_user_attr("elapsed_s", 0.0)
    done, used = len(completed(study)), study.user_attrs.get("elapsed_s", 0.0)
    left_trials, left_time = n_trials - done, timeout_min * 60 - used
    if left_trials > 0 and left_time > 5:
        print(f"[{name}] {done} deneme tamamlanmış → {left_trials} deneme / {left_time/60:.1f} dk kaldı ...")
        last = [time.perf_counter()]
        def save_cb(st, tr):
            now = time.perf_counter()
            st.set_user_attr("elapsed_s", st.user_attrs.get("elapsed_s", 0.0) + (now - last[0])); last[0] = now
            joblib.dump(st, path)
        study.optimize(make_objective(space_fn), n_trials=left_trials, timeout=left_time,
                       callbacks=[save_cb], catch=(Exception,))
        joblib.dump(study, path)
    else:
        print(f"[{name}] checkpoint'ten yüklendi ({done} deneme).")
    c = completed(study)
    print(f"[{name}] tamamlanan deneme: {len(c)} | süre: {study.user_attrs['elapsed_s']/60:.1f} dk | en iyi RMSE: {study.best_value:.3f}")
    return study

def trials_table(study, name):
    t = study.trials_dataframe()
    t = t[t["value"].notna()].copy()
    t.insert(0, "study", name)
    t["best_so_far"] = t["value"].cummin()
    return t

## Aşama 10 — Hyperparameter Optimization (HistGradientBoosting)
### 10.1 Arama uzayı ve gerekçesi

| Parametre | Aralık | Neden aranıyor? |
|---|---|---|
| `window` | {30, 40, 50, 60, 70, 90} | Bölüm 2'de pencere büyüdükçe performans arttı (10 → 20 → 30) ve 30'un üstü denenmemişti. Bu yüzden uzay 30'dan başlayıp yukarı doğru genişletildi. Özellik mühendisliği kararı modelle **birlikte** optimize ediliyor. Not: 30'dan uzun pencerelerde, kısa geçmişi olan motorlarda pencere henüz dolmamış olur. `RollingFeatures` bu durumda mevcut döngülerin tamamını kullanır (`min_periods=1`), yani hesap yine yalnızca geçmişe dayanır. |
| `learning_rate` | 0.01 – 0.3 (log) | Her ağacın katkısı; küçük değerler daha çok ağaç ister ama genelde daha iyi genelleşir |
| `max_iter` | 100 – 800 | Ağaç sayısı; `learning_rate` ile birlikte ayarlanmalı |
| `max_leaf_nodes` | 8 – 64 (log) | Ağaç karmaşıklığı — etkileşimleri yakalama kapasitesi |
| `max_depth` | {None, 4, 6, 8, 12} | Derinlik sınırı (aşırı öğrenmeye karşı) |
| `min_samples_leaf` | 10 – 300 (log) | Yapraktaki minimum örnek. **Bölüm 3, 8.6'daki eğitim–validation farkı (5.7 → 13.2) bu tür düzenlileştirmelerin gerektiğini gösteriyordu.** |
| `l2_regularization` | 0.001 – 10 (log) | Yaprak değerlerine L2 cezası |
| `max_features` | 0.3 – 1.0 | Her bölünmede kullanılan özellik oranı; çok korelasyonlu sensörlerde çeşitlilik sağlar |

**`early_stopping=False` sabitlendi.** HistGradientBoosting 10.000'den fazla satırda iç doğrulama için otomatik olarak verinin %10'unu ayırır. Bu ayırma **satır bazlı ve rastgele** yapılır; yani aynı motorun döngüleri iç doğrulamaya da düşer ve Bölüm 2'de gösterdiğimiz sızıntı burada küçük ölçekte tekrarlanır. Bunun yerine ağaç sayısını (`max_iter`) GroupKFold ile doğrudan arıyoruz.

In [ ]:
def space_hgb(trial):
    window = trial.suggest_categorical("window", WINDOWS)
    est = HistGradientBoostingRegressor(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 100, 800, step=50),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 64, log=True),
        max_depth=trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12]),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 300, log=True),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        early_stopping=False, random_state=SEED)
    return est, window

### 10.2 Tuning öncesi referans
Bölüm 3'teki varsayılan HistGradientBoosting, aynı fonksiyonla yeniden değerlendirilir (pencere = 30).

In [ ]:
r_default = evaluate_config(HistGradientBoostingRegressor(random_state=SEED), 30, train_metrics=True)
DEFAULT_HGB = {k: float(r_default[k].mean()) for k in ["test_RMSE", "test_NASA", "near50_RMSE", "late10_pct", "clip_RMSE", "train_clip_RMSE", "fit_s"]}
DEFAULT_HGB["test_RMSE_std"] = float(r_default["test_RMSE"].std())
print({k: round(v, 3) for k, v in DEFAULT_HGB.items()})

### 10.3 Yöntem 1 — Grid Search (küçük ve anlaşılır)

Grid Search tüm kombinasyonları dener ve parametre sayısı arttıkça maliyeti üstel büyür (8 parametre × 4–5 değer ≈ 100.000+ kombinasyon). Bu yüzden yalnızca **en etkili iki parametre** için küçük bir ızgara kurulur, diğerleri varsayılanda bırakılır. Amaç yöntemin sınırını göstermek.

In [ ]:
grid_path = "checkpoints/grid_hgb.csv"
GRID = ParameterGrid({"learning_rate": [0.03, 0.1], "max_leaf_nodes": [15, 31, 63], "window": [30, 50]})
grid_rows = pd.read_csv(grid_path).to_dict("records") if os.path.exists(grid_path) else []
done_keys = {(r["learning_rate"], r["max_leaf_nodes"], r["window"]) for r in grid_rows}
t0 = time.perf_counter()
for g in GRID:
    if (g["learning_rate"], g["max_leaf_nodes"], g["window"]) in done_keys:
        continue
    est = HistGradientBoostingRegressor(learning_rate=g["learning_rate"], max_leaf_nodes=g["max_leaf_nodes"],
                                        max_iter=300, early_stopping=False, random_state=SEED)
    r = evaluate_config(est, g["window"])
    grid_rows.append({**g, "test_RMSE": r["test_RMSE"].mean(), "test_RMSE_std": r["test_RMSE"].std(), "test_NASA": r["test_NASA"].mean()})
    pd.DataFrame(grid_rows).to_csv(grid_path, index=False)
grid_df = pd.DataFrame(grid_rows).sort_values("test_RMSE")
print(f"Grid: {len(grid_df)} kombinasyon, bu oturumda {time.perf_counter()-t0:.0f} sn")
display(grid_df.round(3))

### 10.4 Yöntem 2 — Random Search ve Yöntem 3 — Bayesian Search (Optuna TPE)

Grid Search'ün tersine bu iki yöntem **8 parametrenin hepsini** aynı anda arar. Aralarındaki tek fark örnekleme stratejisi; arama uzayı ve bütçe aynı:
- **Random Search:** Her deneme bağımsız ve rastgele. Basit ve paralelleştirilebilir, ama geçmiş denemelerden öğrenmez.
- **Bayesian (TPE):** Önceki denemelerin sonuçlarına bakarak "iyi" bölgelerden daha sık örnekler. Aynı bütçede genelde daha iyi sonuca ulaşır.

Random Search de Optuna'nın `RandomSampler`'ı ile yapılır. Böylece iki yöntem **aynı arama uzayı, aynı amaç fonksiyonu ve aynı deneme sayısıyla** birebir karşılaştırılabilir.

In [ ]:
study_hgb_random = run_study("HGB_random", space_hgb, sampler="random")
study_hgb = run_study("HGB_tpe", space_hgb, sampler="tpe")

### 10.5 Yöntemlerin karşılaştırılması

In [ ]:
tt = pd.concat([trials_table(study_hgb_random, "Random Search"), trials_table(study_hgb, "Bayesian (TPE)")])
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
for name, g in tt.groupby("study"):
    ax[0].scatter(g["number"], g["value"], s=15, alpha=.5)
    ax[0].plot(g["number"], g["best_so_far"], lw=2.5, label=f"{name} (en iyi: {g['value'].min():.2f})")
ax[0].axhline(DEFAULT_HGB["test_RMSE"], c="k", ls="--", label=f"varsayılan HGB ({DEFAULT_HGB['test_RMSE']:.2f})")
ax[0].axhline(grid_df["test_RMSE"].min(), c="gray", ls=":", label=f"Grid en iyi ({grid_df['test_RMSE'].min():.2f})")
ax[0].set_xlabel("deneme no"); ax[0].set_ylabel("Test-benzeri RMSE (CV)"); ax[0].set_title("Optimizasyon geçmişi"); ax[0].legend()
ax[0].set_ylim(top=min(tt["value"].max(), DEFAULT_HGB["test_RMSE"] * 1.25))
sns.boxplot(data=tt, x="study", y="value", ax=ax[1]); ax[1].set_title("Deneme skorlarının dağılımı"); ax[1].set_ylabel("RMSE")
plt.tight_layout(); plt.show()

methods = pd.DataFrame({
    "yöntem": ["Grid Search (2 param + pencere)", "Random Search", "Bayesian (TPE)"],
    "deneme": [len(grid_df), len(completed(study_hgb_random)), len(completed(study_hgb))],
    "en_iyi_RMSE": [grid_df["test_RMSE"].min(), study_hgb_random.best_value, study_hgb.best_value],
    "medyan_deneme_RMSE": [grid_df["test_RMSE"].median(), tt[tt.study == "Random Search"]["value"].median(), tt[tt.study == "Bayesian (TPE)"]["value"].median()],
})
display(methods.round(3))

**Nasıl okunmalı?** Kalın çizgi "o ana kadar bulunan en iyi skor". Bayesian aramada skorların zamanla daha iyi bölgede yoğunlaşması (ikinci yarıda noktaların aşağı inmesi) TPE'nin geçmişten öğrendiğini gösterir. Random Search ise uzayı eşit olasılıkla taramaya devam eder. Grid Search yalnızca 2 parametreyi değiştirebildiği için diğer 6 parametrenin varsayılan değerlerine takılı kalır.

> 📝 **Rapor için:** Grafikteki gerçek değerlere bakarak hangi yöntemin hangi deneme sayısında en iyiye ulaştığını yazın. Not: 30 denemelik bütçede Random ile TPE arasındaki fark küçük olabilir. Bu normaldir, TPE'nin avantajı genelde daha uzun aramalarda belirginleşir.

### 10.6 En iyi hiperparametreler ve parametre önemi

In [ ]:
best_hgb = study_hgb if study_hgb.best_value <= study_hgb_random.best_value else study_hgb_random
print("En iyi sonucu veren yöntem:", best_hgb.study_name)
print("En iyi hiperparametreler:")
for k, v in best_hgb.best_params.items():
    print(f"  {k:20s} = {v}")
try:
    imp = optuna.importance.get_param_importances(study_hgb)
    if imp:
        pd.Series(imp).sort_values().plot(kind="barh", figsize=(8, 4), title="Parametre önemi (Bayesian arama, fANOVA)")
        plt.xlabel("RMSE varyansına katkı"); plt.show()
except Exception as e:
    print("Parametre önemi hesaplanamadı:", e)

> 📝 **Rapor için:** Parametre önemi grafiğinde RMSE'yi en çok etkileyen 2–3 parametreyi ve seçilen pencere uzunluğunu yorumlayın (ör. "pencere uzunluğu en etkili parametre → feature engineering kararı modelden daha önemli").

### 10.7 Tuning öncesi ve sonrası

In [ ]:
bt = best_hgb.best_trial
after = {"test_RMSE": bt.value, "test_RMSE_std": bt.user_attrs["test_RMSE_std"], **{k: bt.user_attrs[k] for k in EXTRA_ATTRS}}
cmp = pd.DataFrame({"Varsayılan (Bölüm 3)": {k: DEFAULT_HGB[k] for k in after}, "Tuning sonrası": after})
cmp["değişim_%"] = (cmp["Tuning sonrası"] - cmp["Varsayılan (Bölüm 3)"]) / cmp["Varsayılan (Bölüm 3)"] * 100
display(cmp.round(3))

### 10.8 Seçim yanlılığı kontrolü (selection bias)

Bir modeli 30 farklı ayar arasından **aynı validation noktalarına göre** seçince, en iyi skor biraz "şanslı" olabilir. Model, o belirli kesim noktalarına uymuş da olabilir. Bunu kontrol etmek için en iyi ayarlar **farklı ve daha fazla kesim noktasıyla** (seed = 2024, motor başına 20 nokta) yeniden değerlendirilir. Skor belirgin şekilde kötüleşmiyorsa iyileşme gerçektir.

In [ ]:
def params_to_hgb(p):
    q = {k: v for k, v in p.items() if k != "window"}
    return HistGradientBoostingRegressor(**q, early_stopping=False, random_state=SEED), p["window"]

est, w = params_to_hgb(best_hgb.best_params)
r_new = evaluate_config(est, w, seed=2024, n_cuts=20)
r_def_new = evaluate_config(HistGradientBoostingRegressor(random_state=SEED), 30, seed=2024, n_cuts=20)
print(f"Tuning sonrası — arama noktaları: {best_hgb.best_value:.3f} | yeni noktalar: {r_new['test_RMSE'].mean():.3f} ± {r_new['test_RMSE'].std():.3f}")
print(f"Varsayılan     — arama noktaları: {DEFAULT_HGB['test_RMSE']:.3f} | yeni noktalar: {r_def_new['test_RMSE'].mean():.3f} ± {r_def_new['test_RMSE'].std():.3f}")
print(f"Yeni noktalarda tuning kazancı: {(r_def_new['test_RMSE'].mean() - r_new['test_RMSE'].mean()):.3f} RMSE")

### 10.9 Optimizasyonun maliyeti ve sınırlılıkları

In [ ]:
cost = pd.DataFrame({
    "çalışma": ["Grid Search", "Random Search", "Bayesian (TPE)"],
    "deneme": [len(grid_df), len(completed(study_hgb_random)), len(completed(study_hgb))],
    "toplam_süre_dk": [np.nan, study_hgb_random.user_attrs["elapsed_s"] / 60, study_hgb.user_attrs["elapsed_s"] / 60],
})
cost["model_eğitimi (deneme × 5 fold)"] = cost["deneme"] * 5
display(cost.round(2))

**Sınırlılıklar:**
- **Sınırlı bütçe:** 8 boyutlu bir uzayda 30 deneme, uzayın küçük bir kısmını tarar. Daha uzun bir arama küçük ek kazançlar sağlayabilir.
- **Küçük validation:** 5 fold × 20 motor. Fold'lar arası standart sapma (~1 RMSE), denemeler arasındaki birçok farktan büyük. Bu yüzden 0.1–0.2'lik farklar anlamlı değildir.
- **Seçim yanlılığı:** 10.8'deki kontrolle ölçüldü. Kalan risk, resmi test setinin (Aşama 12) bağımsızlığıyla ayrıca kontrol edilecek.
- **Tek amaç:** Arama yalnızca RMSE'yi minimize etti. NASA skoru (güvenlik) ayrı bir amaç olarak optimize edilmedi, ancak her deneme için kaydedildi ve Aşama 11'deki final kararda kullanılıyor.

## Aşama 11 — Strong Competitor Comparison
### 11.1 Rakiplerin seçimi ve gerekçesi

| Rakip | Neden güçlü bir rakip? |
|---|---|
| **Random Forest** | Bölüm 3'te 2. sıradaydı (fark ~0.4 RMSE). Farklı bir ensemble stratejisi (bagging): ağaçlar bağımsız ve ortalaması alınıyor. Boosting'in aşırı öğrenmeye daha yatkın olduğu durumlarda daha sağlam olabilir. |
| **SVR (RBF)** | Bölüm 3'te 3. sıradaydı; NASA skorunda 2. Tamamen farklı bir aile (kernel). Ağaç tabanlı olmayan en güçlü rakip. |
| **LightGBM** | Adayla aynı aileden (histogram tabanlı gradient boosting), ama yaprak bazlı büyüme, satır/sütun alt örnekleme gibi ek düzenlileştirme seçenekleri var. Endüstride en yaygın kullanılan kütüphanelerden. |
| **XGBoost** | Gradient boosting'in en yerleşik uygulaması. Seviye bazlı büyüme ve farklı düzenlileştirme (`min_child_weight`, L1/L2). |

Yönergenin istediği gibi **rakipler varsayılan parametrelerle bırakılmıyor.** Hepsi adayla **aynı koşullarda** optimize ediliyor:

| Ortak koşul | Değer |
|---|---|
| Arama yöntemi | Optuna TPE, `seed=42` |
| Bütçe | `N_TRIALS` deneme veya `TIMEOUT_MIN` dakika (hangisi önce dolarsa) |
| Amaç fonksiyonu | Aynı (`make_objective` → 5-fold GroupKFold test-benzeri RMSE) |
| Pencere | Aynı aday kümesi {30, 40, 50, 60, 70, 90}, her modelde aranıyor |
| Özellikler, hedef, ölçekleme, fold'lar, kesim noktaları | Aynı |

**Zaman sınırı neden var?** SVR'nin eğitim süresi veri boyutuyla karesel büyüyor; `C` büyüdükçe tek bir deneme 4 dakikayı aşabiliyor. Bu yüzden yalnızca deneme sayısıyla bütçe vermek SVR için saatler sürerdi. Ortak süre sınırı, gerçek hayattaki **hesaplama bütçesi** kısıtını temsil ediyor. Hangi modelin süre dolduğu için daha az deneme yapabildiği raporda açıkça belirtilecek; bu da modelin **maliyetiyle** ilgili bir bulgu.

In [ ]:
def space_rf(trial):
    window = trial.suggest_categorical("window", WINDOWS)
    est = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 300, step=50),
        max_depth=trial.suggest_categorical("max_depth", [None, 10, 16, 24]),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50, log=True),
        max_features=trial.suggest_categorical("max_features", [0.2, 0.33, 0.5, 1.0]),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
        n_jobs=-1, random_state=SEED)
    return est, window

def space_svr(trial):
    window = trial.suggest_categorical("window", WINDOWS)
    est = SVR(kernel="rbf",
              C=trial.suggest_float("C", 0.1, 5.0, log=True),
              epsilon=trial.suggest_float("epsilon", 0.01, 0.5, log=True),
              gamma=trial.suggest_float("gamma", 0.003, 0.1, log=True),
              cache_size=1000)
    return est, window

def space_lgbm(trial):
    window = trial.suggest_categorical("window", WINDOWS)
    est = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 800, step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64, log=True),
        max_depth=trial.suggest_categorical("max_depth", [-1, 4, 6, 8, 12]),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 300, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        random_state=SEED, n_jobs=-1, verbose=-1)
    return est, window

def space_xgb(trial):
    window = trial.suggest_categorical("window", WINDOWS)
    est = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 800, step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        tree_method="hist", random_state=SEED, n_jobs=-1)
    return est, window

**Arama uzaylarının tasarımı:** Her modelde 7–8 parametre ve aynı türden ayarlar aranıyor: karmaşıklık (derinlik/yaprak), düzenlileştirme (min yaprak, L2), alt örnekleme ve pencere. SVR'de `C` üst sınırı 5 tutuldu, çünkü daha büyük `C` değerlerinde tek deneme dakikalar sürüyor. `epsilon` ve `gamma` standartlaştırılmış hedef ve özellik ölçeğine göre seçildi (`gamma="scale"` ≈ 1/56 ≈ 0.018 aralığın ortasında).

### 11.2 Rakiplerin optimizasyonu
Sıra, hızlı modellerden yavaşa doğru; oturum koparsa en az bilgi kaybı olur.

In [ ]:
study_lgbm = run_study("LightGBM_tpe", space_lgbm)
study_xgb = run_study("XGBoost_tpe", space_xgb)
study_rf = run_study("RandomForest_tpe", space_rf)
study_svr = run_study("SVR_tpe", space_svr)

In [ ]:
STUDIES = {"HistGradientBoosting": best_hgb, "LightGBM": study_lgbm, "XGBoost": study_xgb,
           "RandomForest": study_rf, "SVR": study_svr}
budget = pd.DataFrame({n: {"tamamlanan_deneme": len(completed(s)), "süre_dk": s.user_attrs["elapsed_s"] / 60,
                            "ort_deneme_sn": s.user_attrs["elapsed_s"] / max(len(completed(s)), 1),
                            "en_iyi_RMSE (arama)": s.best_value, "seçilen_pencere": s.best_params["window"]}
                       for n, s in STUDIES.items()}).T
budget["bütçe_doldu"] = np.where(budget["tamamlanan_deneme"] >= N_TRIALS, "deneme sayısı", "süre sınırı")
display(budget.round(2))

fig, ax = plt.subplots(figsize=(11, 5))
for n, s in STUDIES.items():
    t = trials_table(s, n)
    ax.plot(t["number"], t["best_so_far"], lw=2, marker=".", label=n)
ax.axhline(DEFAULT_HGB["test_RMSE"], c="k", ls="--", lw=1, label="varsayılan HGB")
ax.set_xlabel("deneme no"); ax.set_ylabel("o ana kadarki en iyi RMSE"); ax.set_title("Tüm modellerin optimizasyon geçmişi"); ax.legend()
plt.show()

> 📝 **Rapor için:** `bütçe_doldu = süre sınırı` olan modeller ayrılan sürede `N_TRIALS` denemeyi tamamlayamadı. Bu, o modelin tuning maliyetinin yüksek olduğunu gösterir ve "kaynak maliyeti değerlendirmesi" başlığında yazılmalı.

### 11.3 Bağımsız yeniden değerlendirme

Tüm modellerin en iyi ayarları, aramada **kullanılmayan** kesim noktalarıyla (seed = 2024, motor başına 20 nokta) yeniden değerlendirilir. Final karşılaştırma bu skorlara dayanır. Böylece arama sırasında "şanslı" çıkan bir model avantaj elde etmez (10.8'deki seçim yanlılığı kontrolünün tüm modellere uygulanmış hali).

In [ ]:
def build_estimator(name, p):
    q = {k: v for k, v in p.items() if k != "window"}
    if name == "HistGradientBoosting": return HistGradientBoostingRegressor(**q, early_stopping=False, random_state=SEED)
    if name == "LightGBM": return LGBMRegressor(**q, subsample_freq=1, random_state=SEED, n_jobs=-1, verbose=-1)
    if name == "XGBoost": return XGBRegressor(**q, tree_method="hist", random_state=SEED, n_jobs=-1)
    if name == "RandomForest": return RandomForestRegressor(**q, n_jobs=-1, random_state=SEED)
    if name == "SVR": return SVR(kernel="rbf", cache_size=1000, **q)
    raise ValueError(name)

re_path = "checkpoints/reevaluation.csv"
re_pred_path = "checkpoints/reevaluation_preds.csv"
re_res = pd.read_csv(re_path) if os.path.exists(re_path) else pd.DataFrame()
re_preds = pd.read_csv(re_pred_path) if os.path.exists(re_pred_path) else pd.DataFrame()
for name, s in {"[varsayılan] HistGradientBoosting": None, **STUDIES}.items():
    if len(re_res) and name in set(re_res["model"]):
        continue
    if s is None:
        est, w = HistGradientBoostingRegressor(random_state=SEED), 30
    else:
        est, w = build_estimator(name, s.best_params), s.best_params["window"]
    t0 = time.perf_counter()
    r, p = evaluate_config(est, w, seed=2024, n_cuts=20, train_metrics=True, model_size=True, return_preds=True)
    r.insert(0, "model", name); p["model"] = name
    re_res = pd.concat([re_res, r], ignore_index=True); re_preds = pd.concat([re_preds, p], ignore_index=True)
    re_res.to_csv(re_path, index=False); re_preds.to_csv(re_pred_path, index=False)
    print(f"{name:36s} RMSE = {r['test_RMSE'].mean():.3f} ± {r['test_RMSE'].std():.3f}  ({time.perf_counter()-t0:.0f} sn)")

### 11.4 Karşılaştırma tablosu

In [ ]:
final_sum = summarize(re_res)[["test_RMSE_mean", "test_RMSE_std", "test_MAE_mean", "test_R2_mean", "test_NASA_mean",
                               "near50_RMSE_mean", "late10_pct_mean", "clip_RMSE_mean", "train_clip_RMSE_mean",
                               "fit_s_mean", "pred_s_mean", "size_MB_mean"]]
final_sum = final_sum.join(budget[["tamamlanan_deneme", "süre_dk", "seçilen_pencere"]], how="left")
final_sum.to_csv("results_stage11_summary.csv")
display(final_sum.round(2))

In [ ]:
order = final_sum.index.tolist()
fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
sns.boxplot(data=re_res, y="model", x="test_RMSE", order=order, ax=ax[0], color="lightsteelblue")
sns.stripplot(data=re_res, y="model", x="test_RMSE", order=order, ax=ax[0], color="k", size=4)
ax[0].set_title("Test-benzeri RMSE — fold dağılımı"); ax[0].set_ylabel("")
ax[1].barh(order, final_sum["test_NASA_mean"], color="salmon"); ax[1].invert_yaxis(); ax[1].set_title("NASA skoru (düşük daha iyi)")
ax[2].scatter(final_sum["fit_s_mean"], final_sum["test_RMSE_mean"], s=np.clip(final_sum["size_MB_mean"], .05, None) * 5 + 50)
for n, r in final_sum.iterrows():
    ax[2].annotate(n, (r["fit_s_mean"], r["test_RMSE_mean"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
ax[2].set_xscale("log"); ax[2].set_xlabel("eğitim süresi / fold (sn, log)"); ax[2].set_ylabel("RMSE"); ax[2].set_title("Performans – maliyet (boyut ∝ MB)")
plt.tight_layout(); plt.show()

### 11.5 Eşleştirilmiş fold karşılaştırması ve arızaya yakın bölge

In [ ]:
piv = re_res.pivot(index="fold", columns="model", values="test_RMSE")
top_model = final_sum.index[0]
paired = pd.DataFrame({
    "ort_fark (rakip − en iyi)": piv.drop(columns=top_model).sub(piv[top_model], axis=0).mean(),
    "std_fark": piv.drop(columns=top_model).sub(piv[top_model], axis=0).std(),
    f"{top_model} daha iyi (fold/5)": piv.drop(columns=top_model).gt(piv[top_model], axis=0).sum(),
}).sort_values("ort_fark (rakip − en iyi)")
print("RMSE'de en iyi:", top_model); display(paired.round(3))

pp = re_preds.copy(); pp["hata"] = pp["RUL_pred"] - pp["RUL_true"]
pp["RUL_aralığı"] = pd.cut(pp["RUL_true"], [0, 25, 50, 75, 100, 125, 150], include_lowest=True)
rb = pp.groupby(["RUL_aralığı", "model"], observed=True)["hata"].apply(lambda e: np.sqrt(np.mean(e ** 2))).unstack()[order]
rb.plot(marker="o", figsize=(11, 4.5), title="Gerçek RUL aralığına göre RMSE"); plt.ylabel("RMSE"); plt.show()
display(rb.round(1))

### 11.6 Final model seçim kuralı (sonuçlara bakmadan önce belirlendi)

Sonuçlara göre kural uydurmamak için seçim kuralı **önceden** tanımlandı ve kodla uygulanıyor:

1. **Birincil ölçüt:** Bağımsız yeniden değerlendirmede (11.3) en düşük ortalama test-benzeri RMSE.
2. **Eşdeğerlik bandı:** RMSE'si en iyi modelden **0.5 birimden az** farklı olan modeller *istatistiksel olarak eşdeğer* kabul edilir. 0.5, fold'lar arası tipik standart sapmanın (~1) yarısı; bu bandın içindeki farklar tesadüf olabilir.
3. **Eşdeğerler arasında:** Önce **güvenlik** (en düşük NASA skoru), NASA skorları da birbirinin %10'u içindeyse **maliyet** (en küçük model boyutu → deploy ve hız) belirleyicidir.

In [ ]:
cands = final_sum.drop(index=[i for i in final_sum.index if i.startswith("[varsayılan]")])
best_rmse = cands["test_RMSE_mean"].min()
equiv = cands[cands["test_RMSE_mean"] <= best_rmse + 0.5].copy()
print("Eşdeğerlik bandındaki modeller (RMSE ≤ en iyi + 0.5):", list(equiv.index))
best_nasa = equiv["test_NASA_mean"].min()
safe = equiv[equiv["test_NASA_mean"] <= best_nasa * 1.10]
FINAL_MODEL = safe.sort_values(["size_MB_mean", "test_RMSE_mean"]).index[0]
print("NASA skoru en iyiye %10 yakın olanlar:", list(safe.index))
print("\n>>> FINAL MODEL:", FINAL_MODEL)
display(final_sum.loc[[FINAL_MODEL]].round(3))

> 📝 **Rapor için — final model gerekçesi:** Yukarıdaki çıktıya göre şu üç soruyu cevaplayan bir paragraf yazın:
> 1. Final model hangi kuralla seçildi? (eşdeğerlik bandı, NASA, boyut)
> 2. Rakiplerden hangi ölçütlerde ayrışıyor? (11.4 tablosu ve 11.5 fold karşılaştırması)
> 3. Kaynak maliyeti nasıl? (eğitim süresi, model boyutu, tuning'de kaç deneme yapabildiği)
>
> Final model aday (HistGradientBoosting) değilse bu bir başarısızlık değildir. Aşama 11'in amacı tam olarak, adayın gerçekten en iyi seçim olup olmadığını adil biçimde sınamak.

### 11.7 Final modelin tüm eğitim verisiyle yeniden eğitilmesi ve kaydedilmesi

Final model, en iyi ayarlarıyla **100 eğitim motorunun tamamı** üzerinde yeniden eğitilir. CV'de her model 80 motorla eğitilmişti; final model mevcut verinin hepsinden öğrenir. Kaydedilen nesne **ham sensör kaydını doğrudan alan tam pipeline**'dır (`RollingFeatures` → `StandardScaler` → model). Aşama 12 ve Streamlit uygulaması bu dosyayı kullanacak.

In [ ]:
fp = STUDIES[FINAL_MODEL].best_params
final_pipe = make_full(build_estimator(FINAL_MODEL, fp), fp["window"])
t0 = time.perf_counter()
final_pipe.fit(df[INDEX_COLS + FEATURES_14], df["RUL"].clip(upper=CAP))
print(f"Final model eğitildi ({time.perf_counter()-t0:.1f} sn)")

# Basit tutarlılık kontrolü: eğitim motorlarının son döngüsünde tahmin ≈ 0 olmalı
last = df.groupby("unit").tail(1)
pred_last = np.clip(final_pipe.predict(df[INDEX_COLS + FEATURES_14]), 0, None)[last.index]
print(f"Eğitim motorlarının son döngüsünde ortalama tahmin: {pred_last.mean():.1f} (beklenen ≈ 0)")

joblib.dump(final_pipe, "final_model.joblib")
meta = {"final_model": FINAL_MODEL, "params": fp, "window": fp["window"], "cap": CAP, "stats": list(STATS),
        "features": FEATURES_14, "cv_test_RMSE": float(final_sum.loc[FINAL_MODEL, "test_RMSE_mean"]),
        "cv_test_RMSE_std": float(final_sum.loc[FINAL_MODEL, "test_RMSE_std"]),
        "cv_test_NASA": float(final_sum.loc[FINAL_MODEL, "test_NASA_mean"]), "versions": VERSIONS,
        "tuning_budget": {"n_trials": N_TRIALS, "timeout_min": TIMEOUT_MIN, "fast_mode": FAST_MODE}}
json.dump(meta, open("final_model_meta.json", "w"), indent=2, ensure_ascii=False, default=str)
config.update({"final_model": FINAL_MODEL, "final_window": fp["window"], "final_params": fp,
               "stage11_final_cv_RMSE": meta["cv_test_RMSE"]})
json.dump(config, open("project_config.json", "w"), indent=2, ensure_ascii=False, default=str)
pd.concat([trials_table(s, n) for n, s in {**STUDIES, "HGB_random": study_hgb_random, "HGB_tpe": study_hgb}.items()]).to_csv("results_stage10_11_trials.csv", index=False)
print("Kaydedildi: final_model.joblib, final_model_meta.json, project_config.json, results_stage10_11_trials.csv, results_stage11_summary.csv")
print(f"Model dosyası boyutu: {os.path.getsize('final_model.joblib')/1e6:.2f} MB")

### 11.8 Özet

| Soru | Nerede? |
|---|---|
| Aday modelin hangi ayarları en iyi? | 10.6 |
| Tuning ne kadar kazandırdı? | 10.7 (aramadaki noktalar) ve 10.8 (bağımsız noktalar) |
| Hangi arama yöntemi daha verimli? | 10.5 |
| Rakipler aynı bütçeyle nereye ulaştı? | 11.2 – 11.4 |
| Final model hangisi ve neden? | 11.6 |
| Maliyeti nedir? | 11.2 (tuning süresi), 11.4 (eğitim süresi, boyut) |

**Sonraki adım (Bölüm 5 — Aşama 12):** `final_model.joblib` ile resmi test seti (`test_FD001.txt` + `RUL_FD001.txt`) **ilk ve tek kez** açılacak; literatür karşılaştırması, hata analizi ve SHAP ile modelin güçlü ve zayıf yönleri raporlanacak.